# Phase 1

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abhinavkum/india-national-family-health-survey-nfhs5")

print("Path to dataset files:", path)

In [ ]:
import os

dataset_path = r"C:\Users\Lenovo\.cache\kagglehub\datasets\abhinavkum\india-national-family-health-survey-nfhs5\versions\3"

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_path}/National Family Health Survey (NFHS-5) 2019-20.csv")

df.head()

In [ ]:
df.columns = [
    "SNo",
    "IndicatorCode",
    "Indicator",
    "SubIndicator",
    "Urban",
    "Rural",
    "Total",
    "NFHS4_Total",
    "State"
]

In [ ]:
df = df[["State", "SubIndicator", "Total"]]
df.head()

In [ ]:
for col in health_df.columns[1:]:
    health_df[col] = (
        health_df[col]
        .astype(str)
        .str.replace(r"[()*]", "", regex=True)
        .replace(["na", "NA", "*"], pd.NA)
    )
    health_df[col] = pd.to_numeric(health_df[col], errors="coerce")

In [ ]:
health_df = selected.pivot(
    index="State",
    columns="SubIndicator",
    values="Total"
).reset_index()

health_df.head()

In [ ]:
health_df = pd.DataFrame()

health_df["State"] = sorted(df["State"].dropna().unique())

In [ ]:
def extract_indicator(keyword):
    temp = df[df["SubIndicator"].str.contains(keyword, case=False, na=False)]
    temp = temp[["State", "Total"]]
    return temp.set_index("State")["Total"]

In [ ]:
temp = df[df["SubIndicator"].str.contains(
    "Men who are overweight or obese",
    case=False,
    na=False
)]

print(temp.shape)
temp.head(20)

In [ ]:
temp["State"].value_counts().head(20)

In [ ]:
def extract_indicator(keyword):
    temp = df[df["SubIndicator"].str.contains(keyword, case=False, na=False)]

    temp = (
        temp[["State", "Total"]]
        .drop_duplicates(subset="State")
        .set_index("State")
    )

    return temp["Total"]

In [ ]:
print(df["State"].value_counts().head(20))

In [ ]:
temp = df[df["SubIndicator"].str.contains(
    "Men who are overweight or obese",
    case=False,
    na=False
)]

print(temp.shape)
print(temp["State"].value_counts().head())

In [ ]:
keywords = [
    "overweight or obese",
    "Children age 6-59 months who are anaemic",
    "All women age 15-49 years who are anaemic",
    "Blood sugar level - high or very high",
    "Elevated blood pressure"
]

filtered = df[
    df["SubIndicator"].str.contains("|".join(keywords), case=False, na=False)
]

print(filtered.shape)
filtered.head()

In [ ]:
health_df = filtered.pivot_table(
    index="State",
    columns="SubIndicator",
    values="Total",
    aggfunc="first"   # handles duplicates automatically
)

health_df = health_df.reset_index()

In [ ]:
print(health_df.columns.tolist())

In [ ]:
health_df = health_df.rename(columns={
    "All women age 15-49 years who are anaemic (%)": "Anaemia_Women",
    "Children age 6-59 months who are anaemic (<11.0 g/dl) (%)": "Anaemia_Children",
    "Blood sugar level - high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%) - Men": "BloodSugar_Men",
    "Blood sugar level - high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%) - Women": "BloodSugar_Women",
    "Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood pressure (%) - Men": "Hypertension_Men",
    "Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood pressure (%) - Women": "Hypertension_Women",
    "Men who are overweight or obese (BMI ≥25.0 kg/m2) (%)": "Obesity_Men",
    "Women who are overweight or obese (BMI ≥25.0 kg/m2) (%)": "Obesity_Women"
})

In [ ]:
health_df = health_df[
    [
        "State",
        "Obesity_Men",
        "Obesity_Women",
        "Anaemia_Children",
        "Anaemia_Women",
        "BloodSugar_Men",
        "BloodSugar_Women",
        "Hypertension_Men",
        "Hypertension_Women"
    ]
]

In [ ]:
import pandas as pd

for col in health_df.columns[1:]:
    health_df[col] = (
        health_df[col]
        .astype(str)
        .str.replace(r"[()*]", "", regex=True)
        .replace(["na", "NA", "*"], pd.NA)
    )
    health_df[col] = pd.to_numeric(health_df[col], errors="coerce")

In [ ]:
health_df["Gender_Obesity_Gap"] = (
    health_df["Obesity_Women"] - health_df["Obesity_Men"]
)

In [ ]:
health_df.head()

In [ ]:
health_df.info()

In [ ]:
health_df.isnull().sum()

# Phase 2

## Task 1

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

top10 = health_df.sort_values(
    by="Hypertension_Women",
    ascending=False
).head(10)

x = np.arange(len(top10))
width = 0.35

plt.figure(figsize=(12,6))

plt.bar(
    x - width/2,
    top10["Hypertension_Men"],
    width,
    label="Men"
)

plt.bar(
    x + width/2,
    top10["Hypertension_Women"],
    width,
    label="Women"
)

plt.xticks(
    x,
    top10["State"],
    rotation=45,
    ha="right"
)

plt.ylabel("Hypertension (%)")
plt.xlabel("State")
plt.title("Top 10 States with Highest Hypertension Rates")

plt.legend()

plt.tight_layout()
plt.show()

## Task 3

# Phase 3

Task 1

In [ ]:
# ============================================================
# TASK 1 : Composite Health Risk Index
# ============================================================

import numpy as np
import pandas as pd

# Make a copy of the cleaned dataframe
risk_df = health_df.copy()

# Health indicators
risk_columns = [
    "Obesity_Men",
    "Obesity_Women",
    "Anaemia_Children",
    "Anaemia_Women",
    "BloodSugar_Men",
    "BloodSugar_Women",
    "Hypertension_Women"
]

# Check missing values
print("Missing Values")
print(risk_df[risk_columns].isnull().sum())

# -----------------------------
# Min-Max Normalization
# -----------------------------

for col in risk_columns:
    risk_df[col] = (
        risk_df[col] - risk_df[col].min()
    ) / (
        risk_df[col].max() - risk_df[col].min()
    )

# -----------------------------
# Composite Health Risk Score
# -----------------------------

risk_df["Composite_Risk"] = (
    risk_df[risk_columns]
    .mean(axis=1)
    * 100
).round(2)

# -----------------------------
# Risk Categories
# -----------------------------

risk_df["Risk_Level"] = np.where(
    risk_df["Composite_Risk"] < 33,
    "Low",
    np.where(
        risk_df["Composite_Risk"] < 66,
        "Medium",
        "High"
    )
)

# -----------------------------
# Ranking
# -----------------------------

risk_df = risk_df.sort_values(
    "Composite_Risk",
    ascending=False
).reset_index(drop=True)

risk_df["Rank"] = risk_df.index + 1

# -----------------------------
# Final Output
# -----------------------------

final_result = risk_df[
    [
        "Rank",
        "State",
        "Composite_Risk",
        "Risk_Level"
    ]
]

display(final_result)

# Save CSV
final_result.to_csv(
    "Composite_Health_Risk_Index.csv",
    index=False
)

print("CSV saved successfully.")

In [ ]:
# ============================================================
# TASK 2 : State Clustering using Graph + BFS
# ============================================================

from collections import deque

# ---------------------------------------------
# Use the dataframe created in Task 1
# ---------------------------------------------

cluster_df = risk_df.copy()

# Threshold for similarity
THRESHOLD = 5

# ---------------------------------------------
# Build Graph (Adjacency List)
# ---------------------------------------------

graph = {}

states = cluster_df["State"].tolist()
scores = cluster_df["Composite_Risk"].tolist()

for i in range(len(states)):
    graph[states[i]] = []

for i in range(len(states)):
    for j in range(i + 1, len(states)):

        if abs(scores[i] - scores[j]) <= THRESHOLD:

            graph[states[i]].append(states[j])
            graph[states[j]].append(states[i])

# ---------------------------------------------
# Breadth First Search
# ---------------------------------------------

visited = set()
clusters = []

for state in graph:

    if state not in visited:

        queue = deque([state])
        visited.add(state)

        current_cluster = []

        while queue:

            node = queue.popleft()
            current_cluster.append(node)

            for neighbour in graph[node]:

                if neighbour not in visited:
                    visited.add(neighbour)
                    queue.append(neighbour)

        clusters.append(current_cluster)

# ---------------------------------------------
# Display Results
# ---------------------------------------------

print("=" * 60)
print("STATE HEALTH CLUSTERS")
print("=" * 60)

for i, cluster in enumerate(clusters, start=1):

    print(f"\nCluster {i}")

    for state in sorted(cluster):
        print("  •", state)

# ---------------------------------------------
# Cluster Summary
# ---------------------------------------------

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"Total States           : {len(states)}")
print(f"Total Clusters Found   : {len(clusters)}")

for i, cluster in enumerate(clusters, start=1):
    print(f"Cluster {i}: {len(cluster)} states")

In [ ]:
# ============================================================
# TASK 3 : Health Atlas using Matplotlib
# ============================================================

import matplotlib.pyplot as plt

# Create a 2x2 grid of plots
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# ----------------------------------------------------------
# Plot 1 : Top 10 Highest Risk States
# ----------------------------------------------------------

top10 = risk_df.nlargest(10, "Composite_Risk")

axes[0, 0].barh(
    top10["State"],
    top10["Composite_Risk"]
)

axes[0, 0].invert_yaxis()
axes[0, 0].set_title("Top 10 Highest Risk States")
axes[0, 0].set_xlabel("Composite Risk Index")
axes[0, 0].set_ylabel("State")

# ----------------------------------------------------------
# Plot 2 : Bottom 10 Lowest Risk States
# ----------------------------------------------------------

bottom10 = risk_df.nsmallest(10, "Composite_Risk")

axes[0, 1].barh(
    bottom10["State"],
    bottom10["Composite_Risk"]
)

axes[0, 1].invert_yaxis()
axes[0, 1].set_title("Top 10 Lowest Risk States")
axes[0, 1].set_xlabel("Composite Risk Index")
axes[0, 1].set_ylabel("State")

# ----------------------------------------------------------
# Plot 3 : Scatter Plot
# Female Obesity vs Female Anaemia
# ----------------------------------------------------------

axes[1, 0].scatter(
    health_df["Obesity_Women"],
    health_df["Anaemia_Women"]
)

axes[1, 0].set_title("Female Obesity vs Female Anaemia")
axes[1, 0].set_xlabel("Female Obesity (%)")
axes[1, 0].set_ylabel("Female Anaemia (%)")

# ----------------------------------------------------------
# Plot 4 : Risk Level Distribution
# ----------------------------------------------------------

risk_counts = risk_df["Risk_Level"].value_counts()

axes[1, 1].pie(
    risk_counts,
    labels=risk_counts.index,
    autopct="%1.1f%%",
    startangle=90
)

axes[1, 1].set_title("Distribution of Risk Levels")

# ----------------------------------------------------------
# Overall Figure
# ----------------------------------------------------------

plt.suptitle(
    "NFHS-5 India Health Atlas",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()

plt.show()

In [ ]:
import json

# Load the GeoJSON file
with open("india_states.geojson", "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

In [ ]:
# ============================================================
# TASK 4 : Interactive Health Risk Dashboard using Folium
# ============================================================

import folium

# ----------------------------------------------------------
# Copy the GeoJSON dataframe mapping from your notebook
# ----------------------------------------------------------

map_df = risk_df.copy()

# ----------------------------------------------------------
# Fix state names to match the GeoJSON
# (Same mappings used in your notebook)
# ----------------------------------------------------------

state_mapping = {
    "Andaman & Nicobar Islands": "Andaman and Nicobar",
    "Dadra & Nagar Haveli and Daman & Diu": "Dadra and Nagar Haveli and Daman and Diu",
    "NCT of Delhi": "Delhi",
    "Jammu & Kashmir": "Jammu and Kashmir",
    "Odisha": "Orissa"
}

map_df["State"] = map_df["State"].replace(state_mapping)

# Remove states if your GeoJSON doesn't contain them
map_df = map_df[
    ~map_df["State"].isin([
        "Lakshadweep",
        "Ladakh"
    ])
]

# ----------------------------------------------------------
# Create Base Map
# ----------------------------------------------------------

india_map = folium.Map(
    location=[22.5, 79],
    zoom_start=5,
    tiles="CartoDB positron"
)

# ----------------------------------------------------------
# Choropleth
# ----------------------------------------------------------

folium.Choropleth(
    geo_data="india_states.geojson",
    data=map_df,
    columns=["State", "Composite_Risk"],
    key_on="feature.properties.NAME_1",
    fill_color="YlOrRd",
    fill_opacity=0.8,
    line_opacity=0.4,
    legend_name="Composite Health Risk Index"
).add_to(india_map)

# ----------------------------------------------------------
# Add Circle Markers
# ----------------------------------------------------------

for _, row in map_df.iterrows():

    state = row["State"]

    geo = geojson_data["features"]

    for feature in geo:

        if feature["properties"]["NAME_1"] == state:

            coords = feature["geometry"]["coordinates"]

            if feature["geometry"]["type"] == "Polygon":

                lon = sum(p[0] for p in coords[0]) / len(coords[0])
                lat = sum(p[1] for p in coords[0]) / len(coords[0])

            else:

                largest = max(coords, key=len)

                lon = sum(p[0] for p in largest[0]) / len(largest[0])
                lat = sum(p[1] for p in largest[0]) / len(largest[0])

            break

    # Marker Color

    if row["Risk_Level"] == "High":
        color = "red"

    elif row["Risk_Level"] == "Medium":
        color = "orange"

    else:
        color = "green"

    popup = f"""
    <b>State:</b> {row['State']}<br>
    <b>Composite Risk:</b> {row['Composite_Risk']:.2f}<br>
    <b>Risk Level:</b> {row['Risk_Level']}
    """

    folium.CircleMarker(
        location=[lat, lon],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=folium.Popup(popup, max_width=250)
    ).add_to(india_map)

# ----------------------------------------------------------
# Save Dashboard
# ----------------------------------------------------------

india_map.save("India_Health_Risk_Dashboard.html")

print("Dashboard saved as India_Health_Risk_Dashboard.html")

india_map